# Stock Buy Decision

**Nhánh C từ 2026-07-17 (SDE_SCORING_REPORT.md, Vòng 3)**: đây là bảng dữ
kiện tham khảo, KHÔNG còn điểm tổng hợp hay khuyến nghị mua/bán -- IC
Newey-West và null-test đều không đạt tiêu chí đã khóa trước cho
`final_score` lẫn từng sub-score. Đọc `stock-decision-engine/SDE_SCORING_REPORT.md`
trước khi dùng bất kỳ số nào dưới đây.

In [3]:
TICKER = "POW"
BENCHMARK = "VNINDEX"
CONFIG_PATH = "../config.yaml"

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
ENGINE_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else Path("stock-decision-engine").resolve()
PROJECT_ROOT = ENGINE_ROOT.parent
SRC = ENGINE_ROOT / "src"
for p in [str(SRC), str(PROJECT_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from data import load_config, load_stock_ohlcv, load_benchmark_ohlcv, load_backtest_evidence
from features import enrich
from simulation import bootstrap_monte_carlo
from scoring import build_facts_table
from report import facts_summary, save_outputs
from visualization import save_all_charts

## Load Data

In [6]:
cfg_path = (ENGINE_ROOT / "notebooks" / CONFIG_PATH).resolve()
cfg = load_config(cfg_path, PROJECT_ROOT)
cfg["ticker"] = TICKER
cfg["benchmark"] = BENCHMARK

stock = load_stock_ohlcv(TICKER, PROJECT_ROOT, cfg["lookback_days"])
benchmark = load_benchmark_ohlcv(BENCHMARK, PROJECT_ROOT, cfg["lookback_days"])
backtest = load_backtest_evidence(TICKER, PROJECT_ROOT)

print(f"Loaded {len(stock):,} stock rows and {len(benchmark):,} benchmark rows.")

Loaded 295 stock rows and 756 benchmark rows.


## Features, Simulation, Score

In [ ]:
stock_f, benchmark_f, relative_strength, rolling_alpha_beta, metrics = enrich(stock, benchmark, cfg)
mc = bootstrap_monte_carlo(stock_f, metrics, cfg)
facts = build_facts_table(metrics, mc.summary, backtest, cfg)

facts["facts"]

## Dashboard Charts

In [ ]:
outputs_dir = ENGINE_ROOT / "outputs"
from IPython.display import Image, display

chart_paths = save_all_charts(
    stock_f,
    benchmark_f,
    relative_strength,
    rolling_alpha_beta,
    mc.paths,
    metrics,
    facts,
    outputs_dir,
    TICKER,
)
for path in chart_paths.values():
    display(Image(filename=str(path)))
chart_paths

## Facts Summary (không phải khuyến nghị -- xem SDE_SCORING_REPORT.md)

In [ ]:
saved = save_outputs(outputs_dir, TICKER, metrics, facts, mc.summary, backtest, root=PROJECT_ROOT)
summary = facts_summary(TICKER, metrics, facts, mc.summary)
print(summary)
saved